In [1]:
from qiskit_qm_provider import QMProvider, add_basic_macros_to_machine, ParameterTable, ParameterPool, IQCCProvider, QMSamplerV2, QMSamplerOptions, QMEstimatorOptions, QMEstimatorV2, QmSaasProvider
from qiskit_qm_provider.job.qua_programs import estimator_program, sampler_program
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit import transpile
import numpy as np
provider = QmSaasProvider()
backend = provider.get_backend("/Users/arthurostrauss/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/Coding_projects/qec_work/quam_state_surface_code")
add_basic_macros_to_machine(backend.machine)
backend.update_target()

2025-12-01 17:32:47,333 - qm - INFO     - Starting session: a130ad79-4648-4cda-bf96-50ef5f4d5003
2025-12-01 17:33:42,131 - qm - INFO     - Performing health check
2025-12-01 17:33:42,399 - qm - INFO     - Cluster healthcheck completed successfully.


/Users/arthurostrauss/Library/CloudStorage/OneDrive-QMMachinesLTD/GitHub/qiskit-qm-provider/qiskit_qm_provider/backend/flux_tunable_transmon_backend.py:56: UserWarning: qiskit.pulse is not available, channel mapping will not be set.
  warnings.warn("qiskit.pulse is not available, channel mapping will not be set.")


In [2]:
use_vpn = False
if use_vpn:
    backend.machine.network["cloud"] = False
    backend.machine.network["port"] = 9510

In [2]:
qc = QuantumCircuit(1)
p = Parameter("p")
qc.rx(p, 0)
qc.measure_all()

transpiled_qc = transpile(qc, backend=backend)
param_values = np.random.rand(20, 10, 1)

In [3]:
ParameterPool.reset()
qm_sampler = QMSamplerV2(backend, QMSamplerOptions(default_shots=500))
job = qm_sampler.run([(transpiled_qc, param_values, 10)])
result = job.result()

2025-12-01 17:34:15,101 - qm - INFO     - Simulating program.


/Users/arthurostrauss/Documents/.venv/lib/python3.13/site-packages/qm/program/_qua_config_schema.py:1880: DeprecationWarning: 'version' is deprecated since "1.2.2" and will be removed in "1.3.0". Please remove it from the Qua config.
  warnings.warn(


QMConnectionError: Encountered connection error from QOP: details: Internal Server Error, status: Status.UNKNOWN

In [8]:
from qiskit_ibm_runtime.fake_provider import FakeJakartaV2
from qiskit_ibm_runtime import EstimatorV2
from qiskit.quantum_info import SparsePauliOp
fake_backend = FakeJakartaV2()
estimator = EstimatorV2(mode=fake_backend)
qc = QuantumCircuit(1)
p = Parameter("p")
# qc.rx(p, 0)
qc.h(0)
new_qc = transpile(qc, backend=fake_backend)
obs = [["X", "Y", "Z"]*3 + ["X"]]*10
obs = [SparsePauliOp(obs_str) for obs_str in obs]
obs_new = [obs_.apply_layout(new_qc.layout) for obs_ in obs]
random_param_values = np.random.rand(20, 10, 1)

# estimator.run([(new_qc, obs_new, random_param_values)])
estimator.run([(new_qc, obs_new,)])

In [9]:
from qiskit.primitives.backend_estimator_v2 import EstimatorPub
# estimator_pub = EstimatorPub.coerce((new_qc, obs_new, random_param_values, 1/np.sqrt(2)))
estimator_pub = EstimatorPub.coerce((new_qc, obs_new, None, 1/np.sqrt(2)))
pub = estimator_pub

In [10]:
estimator_pub.parameter_values

BindingsArray(<shape=(), num_parameters=0>)

In [11]:
estimator_pub.observables

ObservablesArray([{'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}
                  {'IIIIZII': 3.0, 'IIIIXII': 4.0, 'IIIIYII': 3.0}], shape=(10,))

In [12]:
from qiskit_qm_provider.job.qm_estimator_job import _ExecutionPlan
from qiskit_qm_provider import QMEstimatorOptions

execution_plans = _ExecutionPlan.from_pub(estimator_pub, QMEstimatorOptions())

In [ ]:
execution_plans.obs_indices[0]

[(0, 0, 1, 0, 0, 0, 0), (0, 0, 2, 0, 0, 0, 0), (0, 0, 0, 0, 0, 0, 0)]

In [14]:
from collections import defaultdict
circuit = pub.circuit
observables = pub.observables
parameter_values = pub.parameter_values
param_shape = parameter_values.shape
param_indices = np.fromiter(np.ndindex(param_shape), dtype=object).reshape(param_shape)

# 2. Broadcast the parameter indices against the observables.
bc_param_ind, bc_obs = np.broadcast_arrays(param_indices, observables)

# 3. Group observables by unique parameter index.
#    The keys are now tuples of indices, e.g., (0,), (0, 1), etc.
param_obs_map = defaultdict(set)
for index in np.ndindex(*bc_param_ind.shape):
    param_index = bc_param_ind[index]
    param_obs_map[param_index].update(bc_obs[index])

In [15]:
param_obs_map

defaultdict(set, {(): {'IIIIXII', 'IIIIYII', 'IIIIZII'}})

In [17]:
for j, (i1, i2) in enumerate(param_obs_map):
    print(i1, i2)
    print(parameter_values[i1,i2].data) 
    print(parameter_values.ravel()[j].data)

0 0
{('p',): array([0.57052322])}
{('p',): array([0.57052322])}
0 1
{('p',): array([0.47678996])}
{('p',): array([0.47678996])}
0 2
{('p',): array([0.5296302])}
{('p',): array([0.5296302])}
0 3
{('p',): array([0.050148])}
{('p',): array([0.050148])}
0 4
{('p',): array([0.10652145])}
{('p',): array([0.10652145])}
0 5
{('p',): array([0.36056939])}
{('p',): array([0.36056939])}
0 6
{('p',): array([0.51874477])}
{('p',): array([0.51874477])}
0 7
{('p',): array([0.21908478])}
{('p',): array([0.21908478])}
0 8
{('p',): array([0.66790039])}
{('p',): array([0.66790039])}
0 9
{('p',): array([0.13405419])}
{('p',): array([0.13405419])}
1 0
{('p',): array([0.48819974])}
{('p',): array([0.48819974])}
1 1
{('p',): array([0.48307295])}
{('p',): array([0.48307295])}
1 2
{('p',): array([0.09563598])}
{('p',): array([0.09563598])}
1 3
{('p',): array([0.74931419])}
{('p',): array([0.74931419])}
1 4
{('p',): array([0.00722168])}
{('p',): array([0.00722168])}
1 5
{('p',): array([0.127439])}
{('p',): array

In [ ]:
parameter_values.as_array()

array([[[0.93949871],
        [0.95101449],
        [0.70741071],
        [0.06962018],
        [0.15305565],
        [0.77905407],
        [0.73514711],
        [0.31537716],
        [0.18751703],
        [0.28919102]],

       [[0.38871594],
        [0.77363935],
        [0.82483073],
        [0.40120667],
        [0.83107317],
        [0.83407105],
        [0.83396001],
        [0.80607137],
        [0.14543391],
        [0.36285619]],

       [[0.4960332 ],
        [0.85852607],
        [0.69609405],
        [0.72800059],
        [0.33605425],
        [0.95474547],
        [0.12714585],
        [0.18622467],
        [0.40637433],
        [0.22634968]],

       [[0.9890084 ],
        [0.51222301],
        [0.46180568],
        [0.02834884],
        [0.15884132],
        [0.6143646 ],
        [0.08412851],
        [0.26094459],
        [0.05023306],
        [0.61866413]],

       [[0.20842562],
        [0.91495906],
        [0.78335008],
        [0.65580173],
        [0.79872175],
  

In [ ]:
from qiskit_qm_provider.job.qm_estimator_job import observables_to_indices
observables_to_indices(["XX", "YY"])

[(1, 1), (2, 2)]

In [45]:
observables

ObservablesArray([{'IIIIXII': 1.0, 'IIIIYII': 1.0}], shape=(1,))